In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from meeplemate.llm_models import (
    load_tgi_chat_model,
    load_tokenizer,
)

/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
# model_name='teknium/OpenHermes-2.5-Mistral-7B'
model_name='NousResearch/Nous-Hermes-2-SOLAR-10.7B'
tgi_url = "http://tgi:80"

In [3]:
!curl tgi:80/info

{"model_id":"TheBloke/Nous-Hermes-2-SOLAR-10.7B-AWQ","model_sha":"91cc6f03537edcb7f03730d9ca0b44ee118305a9","model_dtype":"torch.float16","model_device_type":"cuda","model_pipeline_tag":"text-generation","max_concurrent_requests":128,"max_best_of":2,"max_stop_sequences":4,"max_input_length":4096,"max_total_tokens":6096,"waiting_served_ratio":1.2,"max_batch_total_tokens":43280,"max_waiting_tokens":20,"max_batch_size":null,"validation_workers":2,"version":"1.4.1","sha":"4139054b82c7eedefcb401400d5ba4172a960ecc","docker_label":"sha-4139054"}

In [4]:
tokenizer = load_tokenizer(model_name)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
chat_model = load_tgi_chat_model(
    tokenizer=tokenizer,
    inference_server_url=tgi_url,
    max_new_tokens=1024,
    timeout=900,
    do_sample=False,
    temperature=0.01,
)

In [6]:
from meeplemate.rag import system_prompt_template

# cot_prompt_template = """\
# Answer the following board game question based on the given rules from the \
# rulebook. Provide your step-by-step reasoning first followed by the answer. \
# Each step should be a separate bullet point. Remember rules found in a board \
# game rulebook generally hold unless there is an explicit exception.

# > Context:
# >>>
# {context}
# >>>
# > Question: {question}
# Answer: Perform a step-by-step deduction based on the given context and \
# provide the answer to the question. Be sure to show every deduction.
# """

cot_prompt_template = """\
Answer the following board game question based on the given rules from the \
rulebook. Provide your step-by-step reasoning first followed by the answer. \
Each step should be a separate bullet point. Remember rules found in a board \
game rulebook generally hold unless there is an explicit exception.

> Context:
>>>
{context}
>>>
> Question: {question}
Walk me through this context in manageable parts step by step, summarizing and \
analyzing as we go. Only afterwards, provide your answer. Ignore any part of \
the context that is not relevant to the question.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_template),
        ("human", cot_prompt_template),
    ],
)

context_when_take_actions = """\
CHARACTER STATS
Each character is basically a collection of weapons, armor, and magic items, with three stats: Level, Race, and Class. For instance, you might describe your character as a “Level 8 Elf Wizard with Boots of Butt-Kicking, a Staff of Napalm, and the Kneepads of Allure.” Level: This is a measure of how generally buff and studly you are. When the rules or cards refer to your Level, capitalized, they mean this number. You gain a level when you kill a monster, or when a card says that you do. You can also sell Items to buy levels (see Items). You lose a level when a card says you do. Your Level can never go below 1. However, your combat strength can be negative, if you get hit by a Curse or suffer some other kind of penalty.
Class: Characters may be Warriors, Wizards, Thieves, or Clerics. If you have no Class card in front of you, you have no class. Yeah, I know, we did that one already.
Each Class has special abilities, shown on the cards. You gain the abilities of a Class the moment you play its card in front of you, and lose them as soon as you discard that card. Some Class abilities are powered by discards. You may discard any card, in play or in your hand, to power a special ability.
See the Class cards for when abilities can be used. Note that a Thief cannot steal while he or the target is fighting - and as soon as a monster is revealed, the fight is on!
You can discard a Class card at any time, even in combat: “I don't wanna be a wizard anymore.” When you discard a Class card, you become classless until you play another Class card.
You may not belong to more than one class at once unless you play the Super Munchkin card.
Race: Characters may be Humans, Elves, Dwarves, or Halflings. If you have no Race card in front of you, you are human.

Race: Characters may be Humans, Elves, Dwarves, or Halflings. If you have no Race card in front of you, you are human.
Humans have no special abilities. The rules for Classes, above, also apply to Races.
You may not belong to more than one race at once unless you play the Half-Breed card.
"""

question = "When can I discard a Race card?"

In [7]:
cot_chain = (prompt | chat_model.bind(temperature=0.4) | StrOutputParser())

print(
    cot_chain.invoke({"context": context_when_take_actions, "question": question})
)


Step 1: Identify the relevant information in the context.
- Race: Characters may be Humans, Elves, Dwarves, or Halflings.
- If you have no Race card in front of you, you are human.
- You may not belong to more than one race at once unless you play the Half-Breed card.

Step 2: Analyze the information.
- The context mentions that you can have a Race card in front of you, representing your chosen race.
- If you don't have a Race card, you are considered human.
- You can only have one race at a time, unless you play the Half-Breed card.

Step 3: Determine if there is any information about discarding Race cards.
- The context does not explicitly mention when you can discard a Race card.

Step 4: Consider the context's information on Classes.
- Class: Characters may be Warriors, Wizards, Thieves, or Clerics.
- You can discard a Class card at any time, even in combat.
- When you discard a Class card, you become classless until you play another Class card.

Step 5: Make a reasonable assumptio

In [8]:
context = """\
In Munchkin, to engage in combat with a monster, compare its combat strength with yours. Combat strength is determined by adding your Level to all positive and negative modifiers from Items and other cards. If the monster's combat strength equals or surpasses yours, you lose the combat and must Run Away (p. 5). If your combat strength exceeds the monster's, you kill it and gain a level (two levels for some large monsters). Additionally, you will acquire the number of Treasures indicated on the monster's card.

Certain cards may enable you to eliminate the monster without defeating it. This is still considered a "win," but you won't receive a level. Unless stated otherwise, you also won't receive the Treasures. If the last monster is removed from a combat, the combat ends immediately.

Some monster cards possess special powers that affect combat, such as a bonus against a specific Race or Class. Always remember to check these.

Both you and other players may use one-shot Treasures or activate Class or Race abilities to aid or hinder you during combat. Some Door cards can also be played in combat, including monster enhancements (see below).

Upon defeating a monster (or monsters), discard the monster(s) and any other played cards, and claim your rewards. However, be aware that someone may play a hostile card on you or utilize a special power right when you believe you have won. When you kill a monster, you must wait a reasonable time, approximately 2.6 seconds, for others to voice any objections. After this waiting period, you have truly killed the monster, and you genuinely receive the levels and Treasures, though they may still complain and argue.

Actions You Can Take
You may perform the following actions at any time:
- Discard a Class or Race card.
- Play a Go Up a Level, Hireling, or Curse card.

You may perform these actions at any time, provided you are not in combat:
- Trade an Item with another player (both players must not be in combat).
- Change which Items you have equipped.
- Play a card you have just received (some cards may be played during combat; see above).

You may perform these actions on your own turn:
- Play a new Class or Race card at any time.
- Sell Items for levels (except when you are in combat).
- Play an Item (most Items cannot be played during combat, but some one-shot Items can; see page 3).

In the game Munchkin, you will navigate through dungeons, defeat monsters, and collect treasures. Your objective is to become the strongest adventurer by leveling up your character. However, be cautious, as other players may try to steal your hard-earned experience points or backstab you to gain levels faster. The ultimate goal is to reach level 10 and become the ultimate Munchkin!

## Treasures

Treasure cards consist of both permanent and "one-shot" cards. Any Treasure card can be placed on the table as soon as it is obtained, or during your own turn at any time, except during combat (unless the rules or the card itself state otherwise).

Faster Game Rules
For a quicker game, you can add a "phase 0" called Listen At The Door. At the start of your turn, before doing anything else, draw a face-down Door card, which you may play or not. Then, arrange cards and Kick Open The Door as normal. If you Loot The Room, draw a face-down Treasure, not a Door.
Usually, a Curse affects its victim immediately (if it can) and is then discarded. However, some Curses give a penalty later in the game or have a continuing effect. Keep these cards until you get rid of the Curse or the penalty takes effect. (Curse cards you keep as a reminder may not be discarded to power Class or Race abilities. Nice try!) You can also allow shared victories — if a player reaches Level 10 in a fight where you are the helper, you also win the game, no matter what Note: If someone plays a “your next combat” Curse on you while you are in Level you are. combat, it counts in that combat! The same is true for a “your next turn” Curse played during your turn.
Game Design by Steve Jackson @ Illustrated by John Kovalic President/Editor-in-Chief: Steve Jackson @ Chief Executive Officer: Philip Reed @ Chief Operating Officer: Susan Bueno @ Chief Creative Officer: Sam Mitschke Munchkin Line Editor: Andrew Hackard @ Production Manager: Sabrina Gonzalez @ Production Artists: Alex Fernandez and Sabrina Gonzalez Project Manager: Darryll Silva @ Operations Manager: Randy Scheunemann @ Director of Sales: Ross Jepson Playtesters: Steve Brinich, Moe Chapman, Paul Chapman, Alain Dawson, Jessie D. Foster, Russell Godwin, Al Griego, Susan Rati, Kat Robertson, and Monica Stephens.
Our deepest thanks to the hundreds, or thousands, or maybe millions of munchkins who have played, told their friends, and suggested new cards since the first release of Munchkin. We love you all and you scare us deeply!
"""

context = """\
Actions You Can Take
You may perform the following actions at any time:
- Discard a Class or Race card.
- Play a Go Up a Level, Hireling, or Curse card.

You may perform these actions at any time, provided you are not in combat:
- Trade an Item with another player (both players must not be in combat).
- Change which Items you have equipped.
- Play a card you have just received (some cards may be played during combat; see above).

You may perform these actions on your own turn:
- Play a new Class or Race card at any time.
- Sell Items for levels (except when you are in combat).
- Play an Item (most Items cannot be played during combat, but some one-shot Items can; see page 3).

In the game Munchkin, you will navigate through dungeons, defeat monsters, and collect treasures. Your objective is to become the strongest adventurer by leveling up your character. However, be cautious, as other players may try to steal your hard-earned experience points or backstab you to gain levels faster. The ultimate goal is to reach level 10 and become the ultimate Munchkin!

## Treasures

Treasure cards consist of both permanent and "one-shot" cards. Any Treasure card can be placed on the table as soon as it is obtained, or during your own turn at any time, except during combat (unless the rules or the card itself state otherwise).
"""


question = "Can I play a Curse while in combat?"

print(
    cot_chain.invoke({"context": context, "question": question})
)

Step 1: Identify the relevant rules for the question.
The question asks if you can play a Curse card while in combat. We need to look for rules that mention playing a Curse card and combat.

Step 2: Analyze the rules for playing a Curse card.
The rulebook states that you can play a Curse card at any time.

Step 3: Analyze the rules for playing cards during combat.
The rulebook states that you may perform the following actions at any time, provided you are not in combat:
- Trade an Item with another player (both players must not be in combat).
- Change which Items you have equipped.
- Play a card you have just received (some cards may be played during combat; see above).

Step 4: Identify if the Curse card can be played during combat.
The rulebook states that some cards may be played during combat, but it does not specifically mention the Curse card.

Step 5: Conclusion
Since the rulebook does not explicitly state whether the Curse card can be played during combat, we cannot definitivel

In [9]:
filter_prompt_temploate = """Given the following question and context, return YES if the context is relevant to the question and NO if it isn't.

> Question: {question}
> Context:
>>>
{context}
>>>
> Relevant (YES / NO):"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", filter_prompt_temploate),
    ],

)

filter_chain = (prompt | chat_model | StrOutputParser())

In [10]:
context = """\
## Treasures

Treasure cards consist of both permanent and "one-shot" cards. Any Treasure card can be placed on the table as soon as it is obtained, or during your own turn at any time, except during combat (unless the rules or the card itself state otherwise).
"""

question = "Can I play a Curse while in combat?"

filter_chain.invoke({"context": context, "question": question})

'YES'

In [11]:
from meeplemate.reword import build_reword_documents_chain
from langchain.schema import Document
from langchain_core.prompts import PromptTemplate

rewrite_prompt_template = """Reword, remove all ambiguity, and make exceedingly clear the following excerpt from a board game rulebook, without changing the meaning or the information conveyed. Use Markdown to format the text.

{context}

Rewritten text:"""

REWRITE_PROMPT = PromptTemplate(
    template=rewrite_prompt_template,
    input_variables=["context"],
)

document_content = """\

In the game Munchkin, you will navigate through dungeons, defeat monsters, and collect treasures. Your objective is to become the strongest adventurer by leveling up your character. However, be cautious, as other players may try to steal your hard-earned experience points or backstab you to gain levels faster. The ultimate goal is to reach level 10 and become the ultimate Munchkin!

## Treasures

Treasure cards consist of both permanent and "one-shot" cards. Any Treasure card can be placed on the table as soon as it is obtained, or during your own turn at any time, except during combat (unless the rules or the card itself state otherwise).
"""

document = Document(page_content=document_content)

reword_chain = build_reword_documents_chain(chat_model.bind(temperature=0.4))

print(reword_chain.invoke([document])[0].page_content)

In the game Munchkin, you will embark on an adventure through dungeons, combat monsters, and gather treasures. Your primary goal is to become the most powerful adventurer by increasing your character's level. However, be mindful, as other players may attempt to steal your hard-earned experience points or backstab you to advance more quickly. The ultimate objective is to reach level 10 and become the ultimate Munchkin!

## Treasure Cards

Treasure cards consist of both permanent and "one-time use" cards. Any Treasure card can be placed on the table as soon as it is acquired, or during your turn at any time, except during combat (unless the rules or the card itself specify otherwise).


In [65]:
summarize_prompt_template = """\
Summarize the following context with respect to the query. Do *NOT* answer the query. Only summarize the relevant information.

> Context:
>>>
{context}
>>>
> Query: {query}
> Summary: \
"""

summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", summarize_prompt_template),
    ],
)

context = """\
In Munchkin, to engage in combat with a monster, compare its combat strength with yours. Combat strength is determined by adding your Level to all positive and negative modifiers from Items and other cards. If the monster's combat strength equals or surpasses yours, you lose the combat and must Run Away (p. 5). If your combat strength exceeds the monster's, you kill it and gain a level (two levels for some large monsters). Additionally, you will acquire the number of Treasures indicated on the monster's card.

Certain cards may enable you to eliminate the monster without defeating it. This is still considered a "win," but you won't receive a level. Unless stated otherwise, you also won't receive the Treasures. If the last monster is removed from a combat, the combat ends immediately.

Some monster cards possess special powers that affect combat, such as a bonus against a specific Race or Class. Always remember to check these.

Both you and other players may use one-shot Treasures or activate Class or Race abilities to aid or hinder you during combat. Some Door cards can also be played in combat, including monster enhancements (see below).

Upon defeating a monster (or monsters), discard the monster(s) and any other played cards, and claim your rewards. However, be aware that someone may play a hostile card on you or utilize a special power right when you believe you have won. When you kill a monster, you must wait a reasonable time, approximately 2.6 seconds, for others to voice any objections. After this waiting period, you have truly killed the monster, and you genuinely receive the levels and Treasures, though they may still complain and argue.

Actions You Can Take
You may perform the following actions at any time:
- Discard a Class or Race card.
- Play a Go Up a Level, Hireling, or Curse card.

You may perform these actions at any time, provided you are not in combat:
- Trade an Item with another player (both players must not be in combat).
- Change which Items you have equipped.
- Play a card you have just received (some cards may be played during combat; see above).

You may perform these actions on your own turn:
- Play a new Class or Race card at any time.
- Sell Items for levels (except when you are in combat).
- Play an Item (most Items cannot be played during combat, but some one-shot Items can; see page 3).

In the game Munchkin, you will navigate through dungeons, defeat monsters, and collect treasures. Your objective is to become the strongest adventurer by leveling up your character. However, be cautious, as other players may try to steal your hard-earned experience points or backstab you to gain levels faster. The ultimate goal is to reach level 10 and become the ultimate Munchkin!

## Treasures

Treasure cards consist of both permanent and "one-shot" cards. Any Treasure card can be placed on the table as soon as it is obtained, or during your own turn at any time, except during combat (unless the rules or the card itself state otherwise).

Faster Game Rules
For a quicker game, you can add a "phase 0" called Listen At The Door. At the start of your turn, before doing anything else, draw a face-down Door card, which you may play or not. Then, arrange cards and Kick Open The Door as normal. If you Loot The Room, draw a face-down Treasure, not a Door.
Usually, a Curse affects its victim immediately (if it can) and is then discarded. However, some Curses give a penalty later in the game or have a continuing effect. Keep these cards until you get rid of the Curse or the penalty takes effect. (Curse cards you keep as a reminder may not be discarded to power Class or Race abilities. Nice try!) You can also allow shared victories — if a player reaches Level 10 in a fight where you are the helper, you also win the game, no matter what Note: If someone plays a “your next combat” Curse on you while you are in Level you are. combat, it counts in that combat! The same is true for a “your next turn” Curse played during your turn.
Game Design by Steve Jackson @ Illustrated by John Kovalic President/Editor-in-Chief: Steve Jackson @ Chief Executive Officer: Philip Reed @ Chief Operating Officer: Susan Bueno @ Chief Creative Officer: Sam Mitschke Munchkin Line Editor: Andrew Hackard @ Production Manager: Sabrina Gonzalez @ Production Artists: Alex Fernandez and Sabrina Gonzalez Project Manager: Darryll Silva @ Operations Manager: Randy Scheunemann @ Director of Sales: Ross Jepson Playtesters: Steve Brinich, Moe Chapman, Paul Chapman, Alain Dawson, Jessie D. Foster, Russell Godwin, Al Griego, Susan Rati, Kat Robertson, and Monica Stephens.
Our deepest thanks to the hundreds, or thousands, or maybe millions of munchkins who have played, told their friends, and suggested new cards since the first release of Munchkin. We love you all and you scare us deeply!
"""

question = "Can I play a Curse while I'm in combat?"

summary_chain = (summary_prompt | chat_model.bind(temperature=0.4) | StrOutputParser())

print(
    summary_chain.invoke({"context": context, "query": question})
)

Yes, you can play a Curse while you're in combat. However, if a player plays a "your next combat" Curse on you while you are in combat, it counts in that combat. The same is true for a "your next turn" Curse played during your turn.


In [52]:
from operator import itemgetter
from meeplemate.consistency import build_run_ntimes_chain
from langchain_core.runnables import RunnablePassthrough


second_prompt_template = """\
Answer the following board game question based on a summary of the rules below:

> Summary:
>>>
{summary}
>>>
> Question: {query}
> Answer: Perform a step-by-step deduction based on the given context and \
provide the answer to the question. Be sure to show every deduction.
"""

second_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", second_prompt_template),
        # ("human", summarize_prompt_template),
        ("assistant", "{summary}"),
        ("human", "Now answer the question: {query}\nLet's think step by step.")
    ],
)

followup_prompt = ChatPromptTemplate.from_messages(
    [
        # ("human", "Now answer the question: {query}\nLet's think step by step."),
        ("assistant", "{cot_answer}"),
        ("human", "So the answer is:")
    ],
)

second_chain = (
    {"context": itemgetter("context"), "query": itemgetter("query"), "summary": summary_chain}
    | RunnablePassthrough.assign(cot_answer=(second_prompt | chat_model.bind(temperature=0.4) | StrOutputParser()))
    | followup_prompt
    | chat_model.bind(temperature=0.01)
    | StrOutputParser()
)
second_chain = build_run_ntimes_chain(second_chain, 10)

for result in second_chain.invoke({"context": context, "query": question}):
    print(result)
    print()


Yes, you can play a Curse card while in combat or at any other point during the game.

Yes, you can play a Curse while you're in combat if the Curse specifies that it affects the next combat or turn.

Yes, you can play a Curse while you're in combat.

Yes, you can play a Curse while you're in combat, as it will count in that combat.

Yes, you can play a Curse while you're in combat if it specifically allows for it or if it is a "your next combat" Curse.

Yes, you can play a Curse while you are in combat.

No, you cannot play a Curse card while you're in combat.

It is likely that playing a Curse card might not be allowed while in combat, but without the specific rules of the game, we cannot provide a definitive answer.

No, you cannot play a Curse while you're in combat. However, if someone else plays a "your next combat" Curse on you while you are in combat, it will count in that combat.

Yes, you can play a Curse while you're in combat, specifically a "your next combat" Curse, and it